# 01 · Data Exploration — BioMassters Dataset

Explore the dataset structure, visualise sample chips, analyse AGB distributions, and characterise missing data patterns.

**Requirements**: Run `python scripts/download_data.py --output-dir data/ --split train` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from biomassters.data.dataset import BioMasstersDataset
from biomassters.utils.visualization import (
    plot_sample, plot_temporal_sequence, make_rgb_composite
)

DATA_ROOT = Path('../data/biomassters')
print('Data root exists:', DATA_ROOT.exists())

In [ ]:
# ── Dataset statistics ──────────────────────────────────────────────────────
ds = BioMasstersDataset(root_dir=DATA_ROOT, split='train')
print(f'Training chips : {len(ds):,}')
print(f'Channels / step: {ds.n_channels} (S1={4}, S2={11})')
print(f'Time steps     : {ds.n_timesteps}')

In [ ]:
# ── Sample visualisation ────────────────────────────────────────────────────
sample = ds[42]
image  = sample['image'].numpy()   # (12, 15, 256, 256)
target = sample['target'].numpy()  # (1, 256, 256)

plot_sample(image, target, month=6, title=f"Chip: {sample['chip_id']}")

In [ ]:
# ── Temporal sequence ────────────────────────────────────────────────────────
plot_temporal_sequence(image, agb_map=target[0])

In [ ]:
# ── AGB distribution ────────────────────────────────────────────────────────
# Collect AGB values from a random subset of chips
import random

agb_values = []
indices = random.sample(range(len(ds)), min(500, len(ds)))
for i in indices:
    t = ds[i]['target'].numpy().ravel()
    agb_values.extend(t[t > 0].tolist())

agb_values = np.array(agb_values)
print(f'AGB statistics (n={len(agb_values):,} valid pixels):')
print(f'  Mean   : {agb_values.mean():.1f} Mg/ha')
print(f'  Median : {np.median(agb_values):.1f} Mg/ha')
print(f'  Std    : {agb_values.std():.1f} Mg/ha')
print(f'  Max    : {agb_values.max():.1f} Mg/ha')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(agb_values, bins=80, color='forestgreen', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('AGB (Mg/ha)', fontsize=12)
axes[0].set_ylabel('Pixel count', fontsize=12)
axes[0].set_title('AGB Distribution (linear scale)', fontweight='bold')

axes[1].hist(np.log1p(agb_values), bins=80, color='darkgreen', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log(1 + AGB)', fontsize=12)
axes[1].set_title('AGB Distribution (log scale)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing data analysis ───────────────────────────────────────────────────
feature_dir = DATA_ROOT / 'train_features'
month_coverage = np.zeros((12, 2))  # [month, modality(S1=0, S2=1)]

chip_ids = [f.name.replace('_S1_00.npy', '') for f in feature_dir.glob('*_S1_00.npy')]
n_sample = min(200, len(chip_ids))
sample_ids = chip_ids[:n_sample]

for chip_id in sample_ids:
    for month in range(12):
        for m_idx, mod in enumerate(['S1', 'S2']):
            p = feature_dir / f'{chip_id}_{mod}_{month:02d}.npy'
            if p.exists():
                month_coverage[month, m_idx] += 1

month_coverage /= n_sample

fig, ax = plt.subplots(figsize=(10, 4))
months = list(range(12))
ax.bar(months, month_coverage[:, 0] * 100, label='Sentinel-1', alpha=0.8, color='steelblue')
ax.bar(months, month_coverage[:, 1] * 100, label='Sentinel-2', alpha=0.6, color='coral', width=0.4)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Coverage (%)', fontsize=12)
ax.set_title('Data Availability by Month', fontweight='bold')
ax.set_xticks(months)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

In [ ]:
# ── Channel correlation heatmap (single chip) ────────────────────────────────
import seaborn as sns

sample = ds[0]
# Use July frame (month 6): (15, 256, 256)
frame = sample['image'][6].numpy().reshape(15, -1).T  # (H*W, 15)

chan_names = [
    'S1 ASC-VV', 'S1 ASC-VH', 'S1 DSC-VV', 'S1 DSC-VH',
    'S2 B2', 'S2 B3', 'S2 B4', 'S2 B5', 'S2 B6',
    'S2 B7', 'S2 B8', 'S2 B8A', 'S2 B11', 'S2 B12', 'S2 CLP'
]

corr = np.corrcoef(frame.T)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, xticklabels=chan_names, yticklabels=chan_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            annot=True, fmt='.1f', annot_kws={'size': 7})
ax.set_title('Channel Correlation (July, single chip)', fontweight='bold')
plt.tight_layout()
plt.show()